---
authors:
  - edesz
date: 2026-05-11
---

# Model Interpretability

## About

We use SHAP to interpret the ML model's predictions for both churn and retention. [As discussed in the project scope](../references/scope/06_analysis.md#model-interpretability), these predictions cover all available customers' data. We will isolate the strongest predictors for each outcome and then cross-reference these SHAP values with our Exploratory Data Analysis (EDA) to evaluate how feature correlations influenced the results.

High correlation between features often obscures model explainability. In tree-based models, correlated variables split the predictive credit that a feature should be given, which dilutes individual SHAP values and makes important drivers appear less significant. By identifying these relationships during EDA, we better accounted for any shifts in feature importance. We will now see the impact of this feature selection.

This interpretability step is important for a ML churn model. The client must understand the reason why predicted churners are leaving to help justify the [retention strategies we recommended in a previous step](./09_estimate_cohort_size_using_savings.ipynb). At the global level, this analysis helps us to go beyond individual customers. It will allow us to identify the ML model's overall decision-making logic. It allows us to identify the systemic drivers of churn across the entire credit-card division. So, instead of fixing one customer's issue, the bank can follow our recommendations to address the high-level problems affecting all the customers.

:::{note}
### Outputs
Charts will be saved in `reports/figures`. Nothing will be exported to the R2 bucket.
:::

## Python Imports

The required Python modules are imported below

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import shap
from dotenv import load_dotenv
from great_tables import GT, loc, md, style

[Initialize required Javascript libraries](https://shap.readthedocs.io/en/latest/generated/shap.plots.initjs.html), so SHAP can be used in a Jupyter notebook

In [ ]:
_ = shap.initjs()

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

Import the custom Python modules are required in this step

In [ ]:
import cc_churn.explanation as shexp
import cc_churn.visualization as vzu
import r2.io_utils as r2io
from utils.df_utils import show_df

## User Inputs

Below we define variables that will be used later

In [ ]:
# R2 data bucket details
# # name of train data key (file) in private R2 bucket
r2_key_train = "train_data.parquet.gzip"
# # name of validation data key (file) in private R2 bucket
r2_key_val = "validation_data.parquet.gzip"
# # name of test data key (file) in private R2 bucket
r2_key_test = "test_data.parquet.gzip"

prefix = "cloud-run"

# trained model prefix
key_prefix = "best_model__"

# predictions prefix
r2_key_pred_prefix = "all_predictions__"

label = "is_churned"

Use environment variables to define an authenticated `boto3` R2 client

In [ ]:
reports_dir = PROJ_ROOT / "reports"
figures_dir = reports_dir / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID_USER2")
secret_access_key = os.getenv("SECRET_ACCESS_KEY_USER2")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

## Load Data

### Trained Model

We will load the best end-to-end ML pipeline

In [ ]:
pipe_best = r2io.joblib_read_latest_r2(
    s3_client,
    bucket_name,
    f"{prefix}/",
    file_pattern=key_prefix,
    file_extension=".joblib",
)

We will now extract the transformer and preprocessor as a pipeline and then extract the best ML model

In [ ]:
pipe_transf_pre = pipe_best.estimator_[:-1]
model = pipe_best.estimator_.named_steps["clf"]

Now we will get the best features that were preprocessed by the best pipeline (`pipe_best`) above

In [ ]:
best_features = (
    pipe_transf_pre.named_steps["pre"]
    .named_transformers_["num"]
    .feature_names_in_.tolist()
)

### Predictions

Load predictions for each customer in all available data

In [ ]:
df = r2io.pandas_read_latest_parquet_r2(
    s3_client,
    bucket_name,
    f"{prefix}/",
    r2_key_pred_prefix,
    ".parquet.gzip",
    None,
).drop(columns=["model_name", "best_decision_threshold"])

This is shown below

In [ ]:
gt = (
    GT(df.sample(5))
    .tab_header(md("**Random Sample of Customers' Features and Predictions**"))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["clientnum"]),
    )
    .tab_style(
        style=style.fill(color="#FFDBBB"),
        locations=loc.body(
            columns=list(
                set(list(df))
                - set(["clientnum", "y_pred", "y_pred_proba", "is_churned"])
            )
        ),
    )
    .tab_style(
        style=[
            style.fill(color="teal"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["y_pred"]),
    )
    .tab_style(
        style=[
            style.fill(color="darkred"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["y_pred_proba"]),
    )
    .fmt_number(columns=["y_pred_proba"], decimals=3)
)
gt

## Separate Features from Target

In [ ]:
X = df.drop(columns=[label])
y = df[label]

Get the features that were not used in training for the best pipeline above

In [ ]:
features_non_preprocessed = list(set(list(X)) - set(best_features))

## Transform Data

We will now apply the custom category combining transformer and then feature pre-processor used during ML development to all customers' data. By doing this, the data is in the same format for ML model training as it would have been during ML development.

Train the custom transformer and preprocessor on the features in all available data

In [ ]:
_ = pipe_transf_pre.fit(X)

Use trained pipeline to get column names after transformation & preprocessing

In [ ]:
columns_transformed = list(
    pipe_transf_pre.get_feature_names_out(input_features=list(X)).tolist()
)

Apply trained transformer to perform category grouping on all data

In [ ]:
X_transformed_preprocessed = pd.DataFrame(
    pipe_transf_pre.transform(X),
    columns=columns_transformed,
    index=X.index,
)
_ = show_df(X_transformed_preprocessed)

Now, we'll combine features that were not pre-processed with the pre-processed numerical features and the true label for all customers

In [ ]:
df_transf_pre = pd.concat(
    [X[features_non_preprocessed], X_transformed_preprocessed, y.rename("y")],
    axis=1,
)[list(X) + ["y"]]

## Model Interpretability

### Customers Predicted to be at Risk of Churning

We will first get the SHAP values (`shap_values`) and SHAP explanation object (`shap_explainer_values_*`) for the customers predicted to be at risk of canceling their credit card services at the bank

In [ ]:
_, shap_explainer_values_pc = shexp.get_shap_values(
    model, df_transf_pre.query("y_pred == 1")[best_features]
)

Plot SHAP summary subplots consisting of

1. (LHS) [beeswarm plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/beeswarm.html#A-simple-beeswarm-summary-plot) that shows how the most important features impact the model's predictions for churners
2. (RHS) [bar plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/bar.html#Global-bar-plot) showing the global feature importance, across all predicted churners

In [ ]:
vzu.plot_shap_summary_plots(
    shap_explainer_values_pc,
    y_axis_annot_offset=0,
    axis_label_fontsize=18,
    y_axis_annot_fontsize=16,
    wspace=0.75,
    fpath=str(figures_dir / "fig_37_interpret_feature_shap_values_churners.png"),
    fig_size=(16, 10),
)

:::{hint}
The width of the bars in the bar plot (right) indicates the importance of the features to the best model's predictions. Bar width is calculated as the average of the absolute value of the individual SHAP values for all predicted churners (customers predicted to cancel their credit card services at the bank).

The beeswarm plot (left) indicates the directionality of the individual contribution to model predictions by each feature: [red indicates a higher contribution while blue indicates a lower contribution](https://doi.org/10.1038/s41598-022-07881-2). Features are sorted top to bottom. Top features are the strongest predictors. One data point is a row in the data. It indicates how the most important features impact the model's predictions for churners. The order of features is based on the average of the absolute value of SHAP values for each feature and so it places more emphasis on broad average impact across all predicted churners. The horizontal position (SHAP value) is important: those feature values to the right of the x=zero (y-axis) axis push a prediction toward the minority class (churn), while feature values to the left push a prediction toward no churn (class 0, the majority class).
:::

**Observations**

1. The three strongest predictors are `total_trans_amt`, `total_revolv_bal`, and `total_ct_chng_q4_q1`, since they have the largest mean absolute SHAP values and therefore the largest average impact on model predictions. For `total_ct_chng_q4_q1`, the blue points are mostly on the right. This means means lower values increase the risk of churn, which matches business intuition that fewer changes in the number of transactions indicate declining activity which leads to churn.

   Similarly for `total_revolv_bal`, lower values appear associated with higher churn contribution. The beeswarm plot (left) shows that many blue points (low feature values) are positioned on the right side of the x-axis, corresponding to positive SHAP values. This indicates that lower revolving balances tend to push predictions toward churn, while higher revolving balances more often reduce churn risk. Our interpretation of this is that customers with very low balances on their credit cards may be using their cards less actively prior to leaving the bank.

   The beeswarm plot also reveals the direction of these effects
   - higher values of `total_trans_amt` (the red points) tend to push predictions toward churn (positive SHAP values)
   - lower values of `total_revolv_bal` (blue points) tend to increase the risk of churn, while higher revolving balances tend to reduce churn risk
   - lower values of `total_ct_chng_q4_q1` strongly increase the likelihood of churn

   These patterns are intuitively reasonable because customers who reduce their transaction count or maintain lower revolving balances may be disengaging from credit card usage before churning.
2. From the beeswarm plot, `total_ct_chng_q4_q1` and `total_amt_chng_q4_q1` seem directionally similar. This points to declining activity which causes a higher risk of churn. `total_amt_chng_q4_q1` is a weaker predictor than `total_ct_chng_q4_q1`, as indicated by its lower mean absolute SHAP value. From EDA, we saw that [these two features have a low-moderate correlation of ~0.38](./03_eda_v2.ipynb#feature-correlation-heatmap), suggesting they capture related but distinct aspects of customer churn behaviour. Lower values of `total_amt_chng_q4_q1` tend to increase churn risk, while higher values weakly reduce it. This direction is consistent with `total_ct_chng_q4_q1`, although the effect size is smaller. By *effect size*, we mean the magnitude of the SHAP contributions for `total_amt_chng_q4_q1` is generally lower than those for `total_ct_chng_q4_q1`. In the beeswarm plot, the horizontal spread of SHAP values for `total_amt_chng_q4_q1` is narrower. This means changes in this feature move the model prediction by a smaller amount on average. So even though both features exhibit similar directional behaviour (declining activity increases churn risk), `total_ct_chng_q4_q1` exerts a stronger influence on the model's final predictions.
3. `credit_limit` and `months_on_book` are comparatively weak predictors, as their SHAP values remain close to zero for most customers. Similarly, `contacts_count_12_mon` and `months_inactive_12_mon`, which are indirect engagement indicators, have smaller overall impacts than transaction-based behavioural features. This suggests that direct measures of credit card usage (i.e. transaction behaviour) and engagement are more important for churn prediction than static account characteristics like support / contact or inactivity behaviour.
4. `total_trans_amt` has a much larger SHAP range than any other feature. This indicates that the best ML model relies heavily on transaction amount when predicting the risk of churn. This spread suggests that this feature captures a lot of nonlinear variation in customer behaviour.
5. Multiple features have non-symmetric SHAP distributions. This suggests the ML model captures nonlinear relationships between customer behaviour and churn risk. As an example, certain ranges of `total_trans_amt` and `total_ct_chng_q4_q1` appear to have disproportionately strong effects on churn prediction. This highlights why tree-based models outperform linear models on this data.

:::{attention}
Correlated variables were excluded using a Pearson correlation > 0.55, per the EDA step. However, these SHAP results suggest the retained behavioural (numerical) features still contain highly informative but also partially overlapping churn signals. This is the case because several transaction-related features exhibit similar directional patterns in their SHAP distributions. For example, lower values of `total_trans_amt`, `total_ct_chng_q4_q1`, and `total_amt_chng_q4_q1` all tend to push predictions toward churn. This indicates that these features capture declining customer engagement.

Even after removing highly correlated features, the remaining features still describe related aspects of customer spending behaviour, transaction frequency, and account usage. This is expected in customer behaviour data because customer activity metrics naturally vary together without being perfectly redundant. The differing SHAP importance magnitudes suggest that each retained feature contributes some unique predictive information beyond the shared churn signal.

Our best ML model was a tree-based model. These ML models handle moderate multicollinearity reasonably well. So, the feature selection likely improved ML model interpretability and helped to reduce both redundancy and ML model training time, without severely reducing the predictive signal in the data.
:::

### Customers Not Predicted to be at Risk of Churning

Next, we'll get the global SHAP values and SHAP explanation object for the customers predicted to be safe (non-churners)

In [ ]:
_, shap_explainer_values_pnc = shexp.get_shap_values(
    model, df_transf_pre.query("y_pred == 0")[best_features]
)

Plot SHAP summary subplots

In [ ]:
vzu.plot_shap_summary_plots(
    shap_explainer_values_pnc,
    y_axis_annot_offset=0,
    axis_label_fontsize=18,
    y_axis_annot_fontsize=16,
    wspace=0.75,
    fpath=str(figures_dir / "fig_38_interpret_feature_shap_values_non_churners.png"),
    fig_size=(16, 10),
)

**Observations**

1. `total_trans_amt` remains the dominant predictor, but the directional pattern reverses. `total_trans_amt` is again the strongest predictor by a large margin, with the highest mean absolute SHAP value (~1.08). However, unlike the churn-risk figure, the beeswarm distribution now shows many blue points (low transaction amounts) on the left side of the x-axis, corresponding to negative SHAP values. This indicates that lower transaction amounts push predictions toward the non-churn class, while higher transaction amounts tend to increase churn predictions less strongly within this subset.

   This is not a perfectly symmetric reversal of the churn-risk plot. For predicted churners, higher SHAP values for `total_trans_amt` strongly pushed predictions toward churn and the spread was much wider (~2.01 mean absolute SHAP value). Here, the influence magnitude is substantially smaller (~1.08) and the distribution is more concentrated around zero. This suggests `total_trans_amt` is still highly influential but extreme transaction behaviour is more characteristic of predicted churners than predicted non-churners
2. For `total_revolv_bal`, many red points (higher revolving balances) appear on the left side of the x-axis, meaning higher balances contribute toward non-churn predictions. By comparison, lower balances (blue points) more often push predictions toward churn. This relationship is more clearly separated than in the churn-risk figure.

   This is consistent with the earlier churn-risk plots. Lower revolving balances there were associated with increased churn risk and higher balances reduced churn risk. Here, the separation appears more stable and concentrated, which suggests the ML model has greater confidence when using revolving balance to identify engaged customers than disengaged customers.
3. `months_inactive_12_mon` is now the third most important predictor (~0.34 mean absolute SHAP value), compared with a weaker role in the churn-risk chart. The beeswarm plot shows that higher inactivity values (red points) tend to push predictions toward churn, while lower inactivity values (blue points) contribute toward non-churn predictions.

    This is directionally opposite to the churn-risk chart and is intuitive when we think about credit-card customer churn. Inactive customers are more likely to churn and active customers are more likely to remain. Inactivity becomes relatively more informative among predicted non-churners than among predicted churners. This may indicate continued activity is a stronger signal of retaining the customers than inactivity on its own being a predictor of churn.
4. Both transaction change features continue to show similar directional behaviour. Lower values (blue points) tend to increase churn risk. Higher values (red points) contribute toward retention. The SHAP distributions are narrower than those in the churn-risk figure, indicating weaker overall influence of this featuer in the non-churners.

   This is not the opposite pattern of that in the plot for predicted churners. Instead, the same directional relationships are visible but with smaller SHAP magnitudes. This suggests the ML model treats declining transaction activity as a stronger signal for churn than increasing activity is for retention.
5. `contacts_count_12_mon` shows one noticeable positive SHAP outlier, where a high number of contacts strongly pushes the prediction toward churning. Most other observations cluster near zero.
                                                                                                       This suggests contacting the bank is not consistently predictive on its own but an unusually high contact frequency may indicate customer dissatisfaction prior to churn. The effect does not appear to be systematic.
6. `credit_limit` and `months_on_book` continue to exhibit SHAP values tightly concentrated around zero, indicating limited predictive importance. This suggests that static account characteristics contribute much less to churn prediction than dynamic behavioural variables. We saw the same for predicted churners.
7. The ML model appears to rely much more heavily on detecting disengagement in credit card customer behavioural signals than signals for customer retention
   - SHAP magnitudes are generally larger for predicted churners than non-churners
   - transaction-related features show wider spreads and stronger positive SHAP values for churn-risk customers
   - non-churn predictions are comparatively more compressed around zero

   These three observations indicate the model is especially sensitive to declining transaction behaviour, reduced engagement with their credit card account and lower revolving balances. All of these are early-warning indicators of customer churn.

## Conclusion

:::{attention} Churners
:class: dropdown
For customers predicted to be at risk of canceling their credit card services at the bank, behavioural transaction features are the dominant drivers of churn prediction. In particular, `total_trans_amt`, `total_revolv_bal`, and `total_ct_chng_q4_q1` have the largest average impact on model output. Lower transaction activity and declining transaction behaviour are strongly associated with an increased risk of churn. Intuitively, this makes sense. By comparison, account-level attributes such as `credit_limit` and `months_on_book` contribute relatively little to churn prediction performance. So, for these customers, the model appears to rely primarily on customer engagement and spending behaviour rather than static customer characteristics.
:::

:::{important} Non-Churners
:class: dropdown
Customers predicted to not be at risk of churning are primarily characterized by stable and active credit card usage behaviour. Transaction-related features such as `total_trans_amt`, `total_revolv_bal`, `total_ct_chng_q4_q1`, and `total_amt_chng_q4_q1` remain the dominant predictors, demonstrating that ongoing engagement with the credit card is central to the ML model's non-churn predictions. In particular, higher revolving balances, lower inactivity, and stable or increasing transaction activity tend to contribute the strongest to predictions of non-churn. This suggests that customers who continue to actively use their credit cards and maintain regular account activity are substantially less likely to churn. The SHAP distributions for predicted non-churners are generally narrower and more concentrated around zero compared with those for predicted churners. This indicates that the model identifies customer retention using more moderate and stable behavioural signals, whereas churn predictions are driven by stronger and more extreme disengagement patterns. Finally, the static account-level features such as `credit_limit` and `months_on_book` remain weak predictors, which confirms our ovreall conclusion that behavioural engagement features are more informative for predicting churn than characteristics of account tenure at the bank.
:::

In summary, across both predicted churners and predicted non-churners, transaction-based behavioural features consistently dominate the ML model's predictions. Features related to transaction amount, transaction count changes, and revolving balance provide stronger predictive signals than static account attributes such as credit limit or tenure. While the directional relationships are generally consistent between these two groups of customers, the churn-risk plots exhibit larger SHAP magnitudes and greater dispersion. This suggests the ML model is more confident when identifying behavioural disengagement patterns associated with churn than when identifying stable retention behaviour.